# Hybrid RAG

This notebook continues from `02_7_vectordb_docker.ipynb` and teaches **hybrid retrieval** by progressively improving the RAG pipeline across three techniques:

1. **Keyword filter** — restrict the candidate pool to documents that contain a specific term
2. **Metadata filter** — use structured fields (score, year, artist) stored natively in ChromaDB
3. **LLM re-ranking** — retrieve a larger set, then let the model promote the most relevant

We run the **same sample query** at each stage so you can compare results directly.

**Prerequisites:** Complete `02_7` first — ChromaDB must be running with the `pitchfork_reviews` collection loaded.

## Setup

In [14]:
%load_ext dotenv

import sys
sys.path.append('../../05_src/')
%dotenv ../../05_src/.secrets
%dotenv ../../05_src/.env

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [15]:
from utils.clients import get_client
from IPython.display import display, Markdown
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import json
import os

client = get_client(use_gateway=False)
MODEL = os.getenv('MODEL')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL')
COLLECTION_NAME = 'pitchfork_reviews'
CHROMA_URL = os.getenv('CHROMA_URL')

### Helper Functions

These functions are brought forward from `02_7`. `get_context_data()` queries ChromaDB and returns enriched result dicts (text + metadata). `generate_prompt()` builds the prompt with album context. `generate_response()` calls the model.

In [16]:
def get_context_data(query: str, collection: chromadb.api.models.Collection, top_n: int,
                     where_document: dict = None, where: dict = None):
    kwargs = dict(query_texts=[query], n_results=top_n)
    if where_document:
        kwargs['where_document'] = where_document
    if where:
        kwargs['where'] = where
    results = collection.query(**kwargs)
    context_data = []
    for idx in range(len(results['ids'][0])):
        details = dict(results['metadatas'][0][idx])
        details['text'] = results['documents'][0][idx]
        context_data.append(details)
    return context_data

def generate_prompt(query: str, context_data: list):
    top_n = len(context_data)
    prompt = f'Given a query, provide a detailed response using the context from relevant Pitchfork reviews. The context will contain references to {top_n} album reviews.\n\n'
    prompt += 'The score is numeric and its scale is from 0 to 10, with 10 being the highest rating. Any album with a score greater than 8.0 is considered a must-listen; album with a score greater than 6.5 is good.\n\n'
    prompt += f'<query>{query}</query>\n\n'
    prompt += '<context>\n'
    for k, context in enumerate(context_data):
        prompt += f'<album {k}>\n'
        prompt += f"- Album Title: {context.get('album', 'N/A')}\n"
        prompt += f"- Album Artist: {context.get('artist', 'N/A')}\n"
        prompt += f"- Album Score: {context.get('score', 'N/A')}\n"
        prompt += f"- Genre: {context.get('genre', 'N/A')}\n"
        prompt += f"- Label: {context.get('label', 'N/A')}\n"
        prompt += f"- Release Year: {context.get('year', 'N/A')}\n"
        prompt += f"- Review Quote: {context.get('text', 'N/A')}\n"
        prompt += f'</album {k}>\n\n'
    prompt += '</context>\n\n'
    prompt += '\nBased on the context and nothing else, provide a detailed response to the query.'
    return prompt

def generate_response(query: str, collection: chromadb.api.models.Collection, top_n: int = 3,
                      where_document: dict = None, where: dict = None):
    context_data = get_context_data(query, collection, top_n,
                                    where_document=where_document, where=where)
    prompt = generate_prompt(query, context_data)
    response = client.responses.create(
        model=MODEL,
        instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
        input=[{'role': 'user', 'content': prompt}],
        max_output_tokens=500,
        temperature=0.7
    )
    return response.output_text

### Connect to ChromaDB

Set `USE_GATEWAY = True` if you are using the API gateway instead of a direct OpenAI key.

In [17]:
USE_GATEWAY = os.getenv('USE_GATEWAY', 'False').lower() == 'true'

chroma = chromadb.HttpClient(host=CHROMA_URL)
if USE_GATEWAY:
    embedding_function = OpenAIEmbeddingFunction(
        api_key='any value',
        api_base='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
        api_type='openai',
        model_name=EMBEDDING_MODEL,
        default_headers={'x-api-key': os.getenv('API_GATEWAY_KEY')}
    )
else:
    embedding_function = OpenAIEmbeddingFunction(
        api_key=os.getenv('OPENAI_API_KEY'),
        model_name=EMBEDDING_MODEL
    )

collection = chroma.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function
)
print(f'Collection loaded: {collection.count():,} documents')

Collection loaded: 14,672 documents


---

## Section 1: Baseline — Pure Vector Search

Before we improve retrieval, let's establish a baseline using the pure vector pipeline from `02_7`. We embed the query, find the 3 most similar review chunks by cosine similarity, and generate a response.

We will reuse the **same query** throughout every section so you can directly compare results.

In [18]:
SAMPLE_QUERY = 'What are some highly rated albums by emerging indie artists?'

In [19]:
baseline_response = generate_response(SAMPLE_QUERY, collection, top_n=3)
display(Markdown(baseline_response))

If you're looking for highly rated albums by emerging indie artists, here are a few standout records from the early 2000s that received strong praises and are definitely worth a listen:

1. **Espers - *Espers* (2004)**
   - **Score:** 8.4
   - **Genre:** Rock
   - **Label:** Locust
   - This album showcases a collective led by Greg Weeks, featuring lush and intricate folk instrumentation. It blends psych-charged melancholia with beautiful melodies, largely due to the ethereal vocals of Meg Baird and the eclectic embellishments from the group's members. The album is a part of a larger trend in indie music rediscovering and revitalizing the folk aesthetics of the late 60s and 70s, making it a significant entry in the indie scene. The review notes the album's depth and its role in the contemporary folk revival, highlighting its artistic richness.

2. **Conductor - *The Comas* (2004)**
   - **Score:** 8.0
   - **Genre:** Rock
   - **Label:** Yep Roc
   - *Conductor* finds The Comas navigating the space between indie and mainstream while maintaining their artistic integrity. The album is noted for its dreamy yet rocking sound, reflecting the complexities of personal experiences, particularly singer Andy Herod’s relationship with actress Michelle Williams. The review emphasizes the album's accessible appeal while also capturing the zeitgeist of indie music in that era. The Comas represent the paradox of indie bands gaining mainstream success while still embodying the spirit of independent music.

3. **Hate - *The Delgados* (2003)**
   - **Score:** 8.1
   - **Genre:** Rock
   - **Label:** Mantra
   - The Delgados are praised for their orchestral pop sound, which combines grandiose melodies with a unique Scottish flair. Their music eschews the typical indie rock tropes, leaning instead into anthemic and resigned melodies that resonate deeply. The review notes the band's ability to create music that's both sophisticated and accessible, positioning them as a key player in the indie rock landscape. This album showcases their evolution from noise-pop to more refined compositions, making it a compelling listen for fans of indie music.

These albums not only highlight the creative diversity within the indie genre but also mark significant contributions from emerging artists who have left a lasting impact on the music scene

**What the baseline does:**
- Converts the query to a vector using the same embedding model as the corpus
- Returns the 3 chunks most similar by cosine distance

**What it doesn't do:**
- It has no knowledge of exact words in the query — `indie` is treated semantically, not literally
- It ignores structured fields like critic score or release year
- It returns the closest chunks by embedding distance, which may not be the *most useful* for the query

The next three sections address each of these gaps.

---

## Section 2: Keyword Filter (`where_document`)

ChromaDB's `where_document={"$contains": keyword}` filter restricts the search to documents whose **text contains the keyword** before similarity ranking is applied.

**When this is useful:**
- The query involves **exact terms** that must appear: artist names, album titles, subgenre labels
- You want **lexical precision** — you need the word to literally be there

**When it hurts:**
- The keyword is too broad or semantically close to the query → no visible difference in results
- The keyword is very rare → too few candidates remain

> **Key insight:** Choose a keyword that is *specific and orthogonal* to what the vector already finds. If your query is already about indie music, filtering by `'indie'` changes nothing — the vector will already retrieve semantically similar content. A narrow subgenre term like `'shoegaze'` or an artist name like `'Arcade Fire'` forces the result set in a new direction.

In [20]:
def keyword_search(query: str, collection: chromadb.api.models.Collection,
                   keyword: str, top_n: int = 3):
    """Vector search restricted to documents containing `keyword`."""
    return get_context_data(
        query, collection, top_n,
        where_document={'$contains': keyword}
    )

In [21]:
keyword_results = keyword_search(SAMPLE_QUERY, collection, keyword='shoegaze', top_n=3)

for i, r in enumerate(keyword_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

In [22]:
keyword_response = generate_response(SAMPLE_QUERY, collection, top_n=3,
                                     where_document={'$contains': 'shoegaze'})
display(Markdown(keyword_response))

**Compare with baseline:** The baseline returned general indie/folk albums. With `keyword='shoegaze'`, every result chunk must literally contain the word *shoegaze* — so the results shift to dream-pop and noise-rock adjacent records where that term appears in the review text.

This illustrates the key trade-off: keyword filtering gives **lexical precision** but misses albums that are *stylistically* shoegaze without being labeled as such.

**Try other keywords:**
- An artist name (`'Arcade Fire'`, `'Radiohead'`) to see only their reviews
- A label name (`'Merge Records'`, `'Matador'`) to filter by imprint
- A technical term (`'saxophone'`, `'ambient'`) to steer toward a different musical texture
- `'indie'` — notice this produces nearly identical results to the baseline, because `indie` is semantically redundant with the query

---

## Section 3: Metadata Filtering (`where`)

When we loaded the collection in `02_7`, we stored structured fields alongside each chunk:

| Field | Type | Example |
|-------|------|---------|
| `score` | float | `8.7` |
| `year` | int | `2001` |
| `artist` | str | `'Radiohead'` |
| `album` | str | `'Kid A'` |
| `genre` | str | `'Electronic, Rock'` |
| `label` | str | `'Capitol'` |

ChromaDB's `where` parameter filters on these fields **before** ranking. Supported operators: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$and`, `$or`, `$in`.

In [23]:
def metadata_search(query: str, collection: chromadb.api.models.Collection,
                    where_filter: dict, top_n: int = 3):
    """Vector search restricted to documents matching `where_filter` on metadata fields."""
    return get_context_data(query, collection, top_n, where=where_filter)

**Example 1: Score filter** — only reviews rated above 8.0

In [24]:
score_results = metadata_search(
    SAMPLE_QUERY, collection,
    where_filter={'score': {'$gt': 8.0}},
    top_n=3
)

for i, r in enumerate(score_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

[0] espers — espers (2004) Score: 8.4
[1] the delgados — hate (2003) Score: 8.1
[2] the delgados — peloton (1999) Score: 8.9


In [25]:
score_response = generate_response(SAMPLE_QUERY, collection, top_n=3,
                                   where={'score': {'$gt': 8.0}})
display(Markdown(score_response))

Here are some highly rated albums by emerging indie artists that you should definitely check out:

1. **Espers – *Espers* (2004)**
   - **Score:** 8.4
   - **Genre:** Rock
   - **Label:** Locust
   - **Overview:** Espers, a folk collective led by Greg Weeks, has crafted a stunning debut that blends elements of psych-folk with lush, baroque melodies. The review highlights the influence of 1960s and 70s folk music, noting how Espers brings a fresh perspective to the genre. The combination of Meg Baird’s ethereal vocals and intricate instrumentation creates a captivating listening experience. The album is praised for its depth and emotional resonance, making it a standout in the indie scene.

2. **The Delgados – *Hate* (2003)**
   - **Score:** 8.1
   - **Genre:** Rock
   - **Label:** Mantra
   - **Overview:** This album showcases The Delgados’ unique ability to blend orchestral elements with rock music, resulting in an engaging sound that goes beyond traditional indie rock. The review appreciates the band’s penchant for anthemic melodies and their Scottish roots, which add a distinctive flavor to their music. The production by Dave Fridmann offers a polished yet invigorating sound, positioning The Delgados as a key player in the indie scene. Their ability to balance grandeur with intimacy makes *Hate* a compelling listen.

3. **The Delgados – *Peloton* (1999)**
   - **Score:** 8.9
   - **Genre:** Rock
   - **Label:** Beggars Banquet
   - **Overview:** A notable entry in the indie rock landscape, this album is recognized for its innovative sound that blends various influences, including the Pixies and Sonic Youth. The review emphasizes the band’s use of orchestral instruments, which enrich the music and enhance the emotional impact of their songwriting. The Delgados’ alternating male and female vocals create a dynamic listening experience, and the album is described as potentially one of the best of its time, solidifying their status as a promising act in the burgeoning Scottish music scene.

These albums not only showcase the talents of emerging indie artists but also highlight the diverse sounds and influences that characterize the genre, making them essential listens for any indie music enthusiast.

**Example 2: Compound filter** — high-scoring albums released from 2000 onwards

In [26]:
compound_results = metadata_search(
    SAMPLE_QUERY, collection,
    where_filter={'$and': [{'score': {'$gt': 8.0}}, {'year': {'$gte': 2000}}]},
    top_n=3
)

for i, r in enumerate(compound_results):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

[0] espers — espers (2004) Score: 8.4
[1] the delgados — hate (2003) Score: 8.1
[2] yo la tengo — and then nothing turned itself inside-out (2000) Score: 8.1


In [27]:
compound_response = generate_response(
    SAMPLE_QUERY, collection, top_n=3,
    where={'$and': [{'score': {'$gt': 8.0}}, {'year': {'$gte': 2000}}]}
)
display(Markdown(compound_response))

If you're looking for highly rated albums by emerging indie artists, there are a couple of standout releases that fit the bill, each showcasing unique qualities that contribute to their acclaim.

1. **Espers - *espers* (2004)**
   - **Score:** 8.4
   - **Genre:** Rock
   - **Label:** Locust
   - This album is a shining example of the modern folk revival, blending psych-charged melancholia with beautifully baroque melodies. The collective, led by Greg Weeks alongside Meg Baird and Brook Sietinsons, offers a lush soundscape that draws from a rich tapestry of late 60s and early 70s folk influences. The review highlights how the band successfully captures the essence of folk music while integrating contemporary indie sensibilities, making it a must-listen for anyone interested in the genre.

2. **The Delgados - *hate* (2003)**
   - **Score:** 8.1
   - **Genre:** Rock
   - **Label:** Mantra
   - The Delgados' *hate* exemplifies what can be described as "indie adult-contemporary," where the band's Scottish roots infuse their anthemic melodies with a unique charm. The production by Dave Fridmann adds a grandiose quality, reminiscent of orchestral pop classics. The review notes that despite a saturation of similar sounds in the indie scene, The Delgados manage to stand out with their emotive songwriting, making this album a significant entry in the emerging indie landscape.

3. **Yo La Tengo - *and then nothing turned itself inside-out* (2000)**
   - **Score:** 8.1
   - **Genre:** Rock
   - **Label:** Matador
   - This album marks a departure for Yo La Tengo, focusing on a more subdued and introspective sound compared to their previous works. The review discusses how the band has stripped away the noise that once characterized their music, opting for a collection of gentle, droning compositions instead. This shift allows listeners to appreciate the band's versatility and depth, showcasing their ability to evolve while still appealing to both underground and mainstream audiences. 

All three albums reflect the vibrancy and diversity of the indie scene, making them essential listens for anyone interested in the genre's evolution.

**A note on genre filtering:**

You might expect to filter by genre with `{'genre': {'$eq': 'Rock'}}`. This works, but only for reviews with *exactly* that genre string. In our dataset, genres are stored as comma-separated strings (e.g., `'Electronic, Rock'`), so `$eq 'Rock'` won't match multi-genre entries.

This is a **schema design trade-off**: storing the primary genre as a separate field (`primary_genre`) would enable clean equality filtering. As a workaround, you can use `artist` or `year` filters (which are single-valued) alongside the semantic query.

**Try it:** Change the `score` threshold or `year` cutoff and observe how the candidate set changes. A very strict filter (e.g., `score > 9.5`) may return no results — the function returns an empty list gracefully.

---

## Section 4: LLM Re-ranking

Vector similarity finds documents that are *semantically close* to the query, but closeness in embedding space doesn't always mean *most useful for answering the question*.

**Re-ranking** separates retrieval from relevance scoring:
1. Retrieve a **larger candidate set** (e.g., top 10) with vector search
2. Ask the language model to **rank candidates by relevance** to the query
3. Return the top-k from the re-ranked list

This adds one API call but can meaningfully improve result quality — especially when the top-1 vector result is semantically close but not the most informative answer.

In [28]:
def llm_rerank(context_data: list, query: str, top_k: int = 3) -> list:
    """Re-rank context_data by relevance to query using a single LLM call."""
    if not context_data:
        return context_data

    # Build a compact candidate list for the model
    candidates_text = ''
    for i, c in enumerate(context_data):
        snippet = c.get('text', '')[:200].replace('\n', ' ')
        candidates_text += (
            f"[{i}] Artist: {c.get('artist', 'N/A')}, "
            f"Album: {c.get('album', 'N/A')}, "
            f"Score: {c.get('score', 'N/A')}, "
            f"Genre: {c.get('genre', 'N/A')}\n"
            f"    Excerpt: {snippet}\n\n"
        )

    rerank_prompt = (
        f'You are ranking album review candidates by how well they answer the user query.\n'
        f'Return ONLY a JSON array of candidate indices ordered from most to least relevant.\n'
        f'Example: [2, 0, 4, 1, 3]\n\n'
        f'Query: {query}\n\n'
        f'Candidates:\n{candidates_text}'
    )

    response = client.responses.create(
        model=MODEL,
        instructions='Return only a valid JSON array of integers. No explanation.',
        input=[{'role': 'user', 'content': rerank_prompt}],
        max_output_tokens=100,
        temperature=0.0
    )

    try:
        raw = response.output_text.strip()
        # Extract the JSON array even if the model wraps it in markdown
        start = raw.find('[')
        end = raw.rfind(']') + 1
        ranked_indices = json.loads(raw[start:end])
        # Filter valid indices and deduplicate
        valid = [i for i in ranked_indices if isinstance(i, int) and 0 <= i < len(context_data)]
        seen = set()
        deduped = [i for i in valid if not (i in seen or seen.add(i))]
        reranked = [context_data[i] for i in deduped]
        # Append any missing candidates at the end
        missing = [c for i, c in enumerate(context_data) if i not in seen]
        return (reranked + missing)[:top_k]
    except Exception:
        # Fallback: return original order
        return context_data[:top_k]

In [29]:
# Retrieve top-10 candidates, then re-rank to top-3
candidates = get_context_data(SAMPLE_QUERY, collection, top_n=10)

print('--- Before re-ranking (top 3 of 10 by vector similarity) ---')
for i, r in enumerate(candidates[:3]):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

reranked = llm_rerank(candidates, SAMPLE_QUERY, top_k=3)

print('\n--- After re-ranking ---')
for i, r in enumerate(reranked):
    print(f"[{i}] {r.get('artist')} — {r.get('album')} ({r.get('year')}) Score: {r.get('score')}")

--- Before re-ranking (top 3 of 10 by vector similarity) ---
[0] the comas — conductor (2004) Score: 8.0
[1] espers — espers (2004) Score: 8.4
[2] the delgados — hate (2003) Score: 8.1

--- After re-ranking ---
[0] the delgados — peloton (1999) Score: 8.9
[1] espers — espers (2004) Score: 8.4
[2] the delgados — hate (2003) Score: 8.1


In [30]:
reranked_response = generate_prompt(SAMPLE_QUERY, reranked)
final_reranked = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': reranked_response}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(final_reranked))

Here are some highly rated albums by emerging indie artists that you should definitely check out:

1. **"peloton" by the delgados** (Score: 8.9)
   - Released in 1999, this album captures the vibrant indie rock scene emerging from Scotland at the time. The Delgados combine elements of rock, pop, and orchestral instrumentation to create a sound that sets them apart from their contemporaries. The review highlights their ability to blend distortion and strong beats with engaging melodies, making "peloton" a remarkable record that showcases their potential as one of rock's promising acts. It’s an album that could easily find its place on many "Best of '99" lists.

2. **"espers" by espers** (Score: 8.4)
   - This self-titled album from 2004 reflects a revival of folk music, blending traditional elements with modern indie aesthetics. The review notes the lush soundscapes created by Greg Weeks and the contributions of members like Meg Baird, whose breathy vocals add depth to the music. The album stands out for its beautiful baroque melodies and psych-charged melancholia, making it a significant work in the indie folk revival movement. It's a testament to how emerging artists can reinterpret classic sounds in fresh and engaging ways.

3. **"hate" by the delgados** (Score: 8.1)
   - Released in 2003, "hate" continues to showcase The Delgados' strength in crafting anthemic melodies and their unique Scottish sensibility. The production by Dave Fridmann adds a lush orchestral quality to the music, which has been described as indie adult-contemporary—a genre that, while sometimes criticized, is celebrated here for its emotional resonance. The review appreciates how the band's evolution from noise-pop to more refined sounds has maintained their distinct charm, making this album another noteworthy entry in their discography.

These albums not only feature strong craftsmanship from emerging indie artists but also highlight the innovative spirit prevalent in the indie music scene. Each record presents a different facet of the genre, promising a rich listening experience for anyone interested in contemporary indie music.

**Cost trade-off:**

Re-ranking adds one extra LLM call — but it is a *short* call: the input is a compact candidate list and the output is just a list of indices. At course scale (10 candidates, ~200 chars each), this is roughly 300–500 tokens, which is negligible.

At production scale with thousands of candidates, you would use a dedicated re-ranking model (e.g., Cohere Rerank, cross-encoders) rather than a generative LLM — but the principle is the same.

---

## Section 5: Combined Pipeline (`hybrid_rag`)

Now we compose all three techniques into a single function:

1. **Keyword filter** (`where_document`) — restrict to documents mentioning the keyword
2. **Metadata filter** (`where`) — enforce quality or recency constraints
3. **LLM re-ranking** — promote the most relevant from the narrowed candidate set

Each layer trades **recall** for **precision**: the keyword and metadata filters reduce the candidate pool, then re-ranking refines the order. The net effect is higher-quality results for queries where the constraints are meaningful.

In [31]:
def hybrid_rag(
    query: str,
    collection: chromadb.api.models.Collection,
    keyword: str = None,
    where_filter: dict = None,
    top_n_candidates: int = 10,
    top_k_final: int = 3
) -> list:
    """
    Hybrid retrieval pipeline:
      1. ChromaDB vector search with optional keyword + metadata filters
      2. LLM re-ranking to top_k_final
    Returns a list of context_data dicts.
    """
    where_document = {'$contains': keyword} if keyword else None
    candidates = get_context_data(
        query, collection, top_n_candidates,
        where_document=where_document,
        where=where_filter
    )
    if not candidates:
        print('Warning: no candidates returned for the given filters.')
        return []
    return llm_rerank(candidates, query, top_k=top_k_final)

**Query 1:** Highly rated indie albums — keyword + score filter + re-ranking

In [32]:
results_1 = hybrid_rag(
    query=SAMPLE_QUERY,
    collection=collection,
    keyword='indie',
    where_filter={'score': {'$gt': 8.0}},
    top_n_candidates=10,
    top_k_final=3
)

response_1 = generate_prompt(SAMPLE_QUERY, results_1)
answer_1 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_1}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(answer_1))

Here are some highly rated albums by emerging indie artists that have garnered significant acclaim:

1. **"last exit" by Junior Boys**  
   - **Score:** 8.9  
   - **Genre:** Rock, Electronic  
   - **Release Year:** 2004  
   - **Overview:** This album stands out for its blend of Sophistipop and electronic elements, showcasing the band's ability to balance danceable beats with introspective lyrics. Tracks like "Teach Me How to Fight" and "Three Words" reveal a depth that combines vulnerability with sonic sophistication. The reviewer highlights the album’s warmth and beauty, suggesting that beneath its sleek production lies a rich emotional landscape. This combination of engaging melodies and poignant themes makes "last exit" a must-listen for fans of innovative indie music.

2. **"what is this heart?" by How to Dress Well**  
   - **Score:** 8.8  
   - **Genre:** Pop/R&B, Electronic  
   - **Release Year:** 2014  
   - **Overview:** How to Dress Well, the project of Tom Krell, pushes the boundaries of indie R&B with this album. It’s characterized by its introspective lyrics that explore themes of love and loss, as well as complex production that melds electronic sounds with emotional depth. The review emphasizes Krell's ability to articulate the nuances of relationships, drawing parallels to broader themes of change and desire. His approach not only reflects personal struggles but also resonates with the zeitgeist of contemporary indie music, marking it as a significant contribution to the genre.

3. **"Urban Renewal Program" by Various Artists**  
   - **Score:** 8.2  
   - **Genre:** Various (compilation)  
   - **Release Year:** 2002  
   - **Overview:** Compiled by the Chicago label Chocolate Industries, this album features a range of underground artists and highlights the label's rising influence in the indie scene. The compilation is noted for its blend of hip-hop and producer-oriented instrumental tracks, with contributions from notable figures like Aesop Rock and Prefuse 73. The reviewer praises the collection for its cohesive sound and the way it captures a vibrant underground culture, making it a noteworthy listen for those interested in the intersections of hip-hop and indie music.

These albums not only received high praise from Pitchfork but also represent a diverse array of sounds and themes within the indie music landscape, making them essential listens for fans of the

**Query 2:** Classic albums from the 90s — year filter + re-ranking

In [33]:
query_2 = 'What are the most influential rock albums of the 1990s?'

results_2 = hybrid_rag(
    query=query_2,
    collection=collection,
    keyword='rock',
    where_filter={'$and': [{'score': {'$gt': 7.5}}, {'year': {'$gte': 1990}}, {'year': {'$lt': 2000}}]},
    top_n_candidates=10,
    top_k_final=3
)

response_2 = generate_prompt(query_2, results_2)
answer_2 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_2}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(answer_2))

The 1990s were a transformative decade in rock music, marked by a diverse range of influential albums that pushed boundaries and redefined the genre. Two standout albums from this era that are often highlighted for their impact are *Emergency & I* by The Dismemberment Plan and *Califone* by Califone, both released in 1999.

1. **Emergency & I by The Dismemberment Plan (Score: 9.6)**:
   This album is considered a landmark in the rock genre, blending elements of dance-punk and indie rock while offering a return to intelligent and narrative-driven songwriting. The Dismemberment Plan's unique sound incorporates influences from hip-hop and soul, creating a rich tapestry of emotions that resonate with listeners, especially those in their twenties. Tracks like "Spider in the Snow" and "Gyroscope" exemplify this emotional depth, capturing feelings of confusion, loss, and the complexities of post-collegiate life. The album's ability to transform concert crowds into cathartic experiences further solidifies its influence. Its innovative approach to rock, described as "the new new wave," has made it a seminal work that continues to inspire artists today.

2. **Califone by Califone (Score: 9.2)**:
   Another pivotal album from the late 90s, *Califone* showcases an evolution in rock by integrating the noise and experimental elements of electronic music in a fresh and organic way. This album stands out for its ability to blend various genres while maintaining a cohesive sound that resonates with listeners. The review suggests that *Califone* not only offers compelling music for the present but also serves as a touchstone for future generations, making it a must-listen for rock enthusiasts. Its forward-thinking style has influenced countless artists who seek to push the boundaries of rock music.

Both *Emergency & I* and *Califone* exemplify the innovative spirit of the 1990s rock scene, with their respective sounds and lyrical depth leaving a lasting impact on the genre. These albums are not just significant for their time; they continue to be relevant and influential, marking them as essential listens for anyone interested in the evolution of rock music during this transformative decade.

**Query 3:** Electronic music — keyword only + re-ranking (no metadata filter)

In [34]:
query_3 = 'Recommend some groundbreaking electronic albums with experimental production.'

results_3 = hybrid_rag(
    query=query_3,
    collection=collection,
    keyword='electronic',
    where_filter=None,
    top_n_candidates=10,
    top_k_final=3
)

response_3 = generate_prompt(query_3, results_3)
answer_3 = client.responses.create(
    model=MODEL,
    instructions='You are a helpful assistant that provides information based on Pitchfork reviews.',
    input=[{'role': 'user', 'content': response_3}],
    max_output_tokens=500,
    temperature=0.7
).output_text
display(Markdown(answer_3))

If you're looking for groundbreaking electronic albums that feature experimental production, here are three standout recommendations that you shouldn't miss:

1. **"Flatland" by Objekt (2014)** - Scoring an impressive **8.0**, Objekt's debut LP is a deep dive into the current state of electronic music. The album is a reflection of the evolving landscape, where traditional definitions of the genre are increasingly blurred. The review highlights Objekt's technical prowess, stemming from his background in electronic and information engineering, which translates into his meticulous yet impulsive production style. The tracks are experimental at their core, showcasing a blend of clinical precision and raw emotion, making "Flatland" a significant work that challenges the boundaries of electronic music.

2. **"Artificial Intelligence" by Various Artists (2005)** - This compilation, with a high score of **8.8**, captures the essence of "electronic listening music" from the early '90s. Although some tracks might seem less memorable by today's standards, the compilation features profound contributions from artists like Autechre, whose track "Chatter" fuses various electronic styles into a lush soundscape. The review notes that the compilation serves as a historic reminder of the potential for originality in electronic music, echoing the sentiment that it may have been easier to dream of groundbreaking ideas back in the early days of the genre. The album is a testament to the creative spirit that has always been central to electronic music.

3. **"4 Parabolic Mixes" by Philip Jeck, Main, Oval, Henri Pousseur (2003)** - While this album scores a **7.3**, it offers a fascinating exploration of electronic music through the lens of historical experimentation. The review discusses the evolution of electronic music, referencing early pioneers and how their innovations laid the groundwork for contemporary sounds. This album features reworkings of Henri Pousseur's earlier compositions, illustrating the development of electronic music as an art form. It may appeal to those interested in the historical context and theoretical aspects of electronic music, providing a reflective experience that complements the more mainstream offerings.

Each of these albums showcases different facets of experimental production within the electronic genre, offering listeners a rich tapestry of sound that is both innovative and thought-provoking.

---

## Summary

| Technique | What it does | Best for | Risk |
|-----------|-------------|----------|------|
| **Pure vector** | Semantic similarity across full corpus | Open-ended queries | May retrieve semantically close but unhelpful chunks |
| **Keyword filter** | Restricts to documents containing an exact term | Artist names, genre labels, specific albums | Misses synonyms; empty result if keyword is rare |
| **Metadata filter** | Restricts to chunks meeting numeric/string criteria | Quality gates (score), time ranges (year) | Tight filters reduce recall |
| **LLM re-ranking** | Model re-orders a candidate pool by relevance | Refining any of the above | Extra API call; negligible cost at small scale |
| **Hybrid (all three)** | Combines all layers | High-precision queries with clear constraints | Very tight keyword + metadata may return no candidates |

**Next steps:**
- Notebook `02_9` (coming soon): quantitative evaluation — measure how much each technique improves hit rate
- Try query intent extraction: use the LLM to parse `genre`, `score`, and `year` constraints from a natural-language query, then pass them as `where_filter` automatically